In [ ]:
# Copyright 2026 50Hertz Transmission GmbH and Elia Transmission Belgium SA/NV
#
# This Source Code Form is subject to the terms of the Mozilla Public License, v. 2.0.
# If a copy of the MPL was not distributed with this file,
# you can obtain one at https://mozilla.org/MPL/2.0/.
# Mozilla Public License, version 2.0

# Example 4: the data contracts, package by package

`example3` runs the same grid through the *pipeline* helpers. This notebook does the same journey
with the packages called **directly**, so that every hand-over between them is visible: what object
is built, who writes it, and what the next package receives.

Four stages, four contracts:

| Stage | Package | Input contract | Produces |
|---|---|---|---|
| 1. Import | `toop_engine_importer` | `CgmesImporterParameters` (+ the business contingency list) | `nminus1_definition.json` — **canonical** |
| 2. DC preprocessing | `toop_engine_dc_solver` | the canonical definition + `PreprocessParameters` | `StaticInformation`, `NetworkData`, `dc_nminus1_definition.json` — **DC projection** |
| 3. Topology optimization | `toop_engine_topology_optimizer` | `DCOptimizerParameters` + `StaticInformation` | candidate topologies |
| 4. AC validation | `toop_engine_contingency_analysis` | the canonical definition, sanitised in memory | `LoadflowResultsPolars` |

The full contract description lives in `docs/importer/complex_contingency_import_handoff.md`.

In [ ]:
import logging
from pathlib import Path
import tempfile

import pypowsybl
import structlog

# Preprocessing is chatty at info level; keep warnings, which is where the contract
# diagnostics (unsupported elements, dropped contingencies) surface.
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))
from fsspec.implementations.dirfs import DirFileSystem

from toop_engine_interfaces.folder_structure import PREPROCESSING_PATHS
from toop_engine_interfaces.messages.preprocess.preprocess_commands import (
    AreaSettings,
    CgmesImporterParameters,
    PreprocessParameters,
)
from toop_engine_interfaces.nminus1_definition import (
    copy_without_spps_rules,
    copy_without_switch_only_contingencies,
    load_nminus1_definition,
)

FIXTURE = Path("../data/complex_grid")
GRID_FILE = FIXTURE / "grid.xiidm"
CONTINGENCY_LIST = FIXTURE / "contingency_list_complex.json"

# The importer writes into data_folder; keep the checked-in fixture read-only.
WORK = Path(tempfile.mkdtemp(prefix="toop_example4_"))
print("grid            :", GRID_FILE)
print("contingency list:", CONTINGENCY_LIST)
print("work folder     :", WORK)

## Stage 1 — Importer

`CgmesImporterParameters` is the **input contract**. `schema_format` selects the parser for the
business contingency list; `area_settings` decides which part of the grid is monitored, controllable
and outaged.

`convert_file` writes the processed grid folder, including the **canonical** N-1 definition.

In [ ]:
from toop_engine_importer.pypowsybl_import import preprocessing

importer_parameters = CgmesImporterParameters(
    grid_model_file=GRID_FILE,
    data_folder=WORK,
    contingency_list_file=CONTINGENCY_LIST,
    schema_format="ContingencyImportSchemaComplex",   # <- selects the grouped-contingency parser
    area_settings=AreaSettings(
        cutoff_voltage=1.0,
        control_area=["BE", "NL"],
        view_area=["BE", "NL"],
        nminus1_area=["BE", "NL"],
    ),
)

import_result = preprocessing.convert_file(importer_parameters=importer_parameters)
print(type(import_result).__name__, "->", sorted(p.name for p in WORK.iterdir()))

### The canonical definition

`nminus1_definition.json` is what every downstream package reads. The importer is its owner — DC and
CA never modify it. Note that a grouped case keeps **all** its source elements, including the
switches that isolate the faulted component, and that the SPPS rules survive import.

In [ ]:
canonical = load_nminus1_definition(WORK / PREPROCESSING_PATHS["nminus1_definition_file_path"])

print(f"source_schema     : {canonical.source_schema}")
print(f"id_type           : {canonical.id_type}")
print(f"monitored elements: {len(canonical.monitored_elements)}")
print(f"SPPS rules        : {[rule.scheme_name for rule in (canonical.spps_rules or [])]}")
print(f"contingencies     : {len(canonical.contingencies)}\n")

for contingency in canonical.contingencies:
    kinds = ", ".join(f"{element.id}({element.type})" for element in contingency.elements[:3])
    more = f" (+{len(contingency.elements) - 3})" if len(contingency.elements) > 3 else ""
    print(f"  {contingency.id:<32} n={len(contingency.elements)}  {kinds}{more}")

## Stage 2 — DC preprocessing

`load_grid` builds the solver inputs. It reads the **canonical** definition, projects it onto what DC
can actually compute, and writes that projection to `dc_nminus1_definition.json`.

The projection is an *output*: DC never reads it back as input, otherwise every run would re-project
an already-projected definition.

In [ ]:
from toop_engine_dc_solver.preprocess.convert_to_jax import load_grid

stats, static_information, network_data = load_grid(
    data_folder_dirfs=DirFileSystem(str(WORK)),
    pandapower=False,
    parameters=PreprocessParameters(action_set_clip=2**10),
)

dc_definition = load_nminus1_definition(WORK / PREPROCESSING_PATHS["dc_nminus1_definition_file_path"])

canonical_ids = [c.id for c in canonical.contingencies]
dc_ids = [c.id for c in dc_definition.contingencies]

print(f"canonical : {len(canonical_ids):2d} {canonical_ids}")
print(f"projection: {len(dc_ids):2d} {dc_ids}")
print(f"dropped   : {[i for i in canonical_ids if i not in dc_ids]}")
print(f"SPPS in projection: {dc_definition.spps_rules}")

### Why cases are dropped, and how the rest are classified

DC classifies a case by the elements that **survive** projection, not by how many it started with.
A line plus the two breakers that isolate it *is* a single branch outage; a three-winding transformer
expands to three legs and is a genuine multi-outage. A case with nothing DC can represent is dropped
with a diagnostic.

`contingency_ids` is ordered `branch -> multi-outage -> injections`, and validation compares solver
output against it **positionally**, so this order is a contract in its own right.

In [ ]:
single = [
    network_data.contingency_id_by_element_id.get(branch_id, branch_id)
    for branch_id, outaged in zip(network_data.branch_ids, network_data.outaged_branch_mask, strict=True)
    if outaged
]
print("single branch outages:", single)

for multi_id, mask in zip(network_data.multi_outage_ids, network_data.multi_outage_branch_mask, strict=True):
    members = [b for b, m in zip(network_data.branch_ids, mask, strict=True) if m]
    print(f"multi-outage {multi_id}: {members}")

print("\ncontingency_ids (solver order):", network_data.contingency_ids)
print("static_information agrees      :",
      list(static_information.solver_config.contingency_ids) == network_data.contingency_ids)

## Stage 3 — Topology optimization (DC)

The optimizer's contract is `DCOptimizerParameters` plus the preprocessed folder: it loads
`static_information.hdf5` and `action_set.json` itself, addressed by a `GridFile` relative to
`processed_gridfile_fs`. Its search space is the set of substations that have actions.

On this grid the imported list leaves none. DC keeps 4 of the 8 cases, and the reduced branch
dimension then leaves every relevant substation without a split to try — preprocessing logs
`Relevant sub has no actions`. The optimizer contract does not depend on which definition produced
the folder, so we import the **same grid network-derived** (no `contingency_list_file`) and optimize
that. It doubles as a look at the second definition contract: the mask-derived one.

In [ ]:
print("actionable substations with the imported list:",
      len(static_information.dynamic_information.action_set.action_start_indices))

# Same grid, network-derived definition -> an action set the optimizer can search.
# That definition declares one contingency per switch, so DC drops ~570 of them and logs a line
# each; the diagnostic was already visible in Stage 2, so quiet it for this second import.
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.ERROR))

OPT_WORK = Path(tempfile.mkdtemp(prefix="toop_example4_opt_"))
preprocessing.convert_file(
    importer_parameters=CgmesImporterParameters(
        grid_model_file=GRID_FILE,
        data_folder=OPT_WORK,
        area_settings=AreaSettings(
            cutoff_voltage=1.0,
            control_area=["BE", "NL"],
            view_area=["BE", "NL"],
            nminus1_area=["BE", "NL"],
        ),
    )
)
_stats, opt_static_information, opt_network_data = load_grid(
    data_folder_dirfs=DirFileSystem(str(OPT_WORK)),
    pandapower=False,
    parameters=PreprocessParameters(action_set_clip=2**10),
)
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

mask_derived = load_nminus1_definition(OPT_WORK / PREPROCESSING_PATHS["nminus1_definition_file_path"])

print(f"mask-derived definition : {len(mask_derived.contingencies)} contingencies "
      f"(source_schema={mask_derived.source_schema})")
print("actionable substations  :",
      len(opt_static_information.dynamic_information.action_set.action_start_indices))

### Running the optimization

`initialize_optimization` evaluates the unsplit baseline, then each `run_epoch` consumes one chunk of
the candidate workset. A complete run keeps going until the workset is exhausted, so we loop rather
than calling a single epoch — with a wall-clock budget as the guard for bigger grids.

In [ ]:
import time

from toop_engine_topology_optimizer.dc_bruteforce.optimizer import (
    get_num_branch_topologies_tried,
    initialize_optimization,
    is_exhausted,
    run_epoch,
)
from toop_engine_topology_optimizer.interfaces.messages.commons import Framework, GridFile
from toop_engine_topology_optimizer.interfaces.messages.dc_params import (
    BatchedMEParameters,
    DCOptimizerParameters,
    DescriptorDef,
    LoadflowSolverParameters,
)

grid_file = GridFile(framework=Framework.PYPOWSYBL, grid_folder=OPT_WORK.name)
dc_params = DCOptimizerParameters(
    summary_frequency=1,
    check_command_frequency=1,
    ga_config=BatchedMEParameters(
        runtime_seconds=60,
        iterations_per_epoch=8,
        me_descriptors=(DescriptorDef(metric="split_subs", num_cells=2),),
    ),
    loadflow_solver_config=LoadflowSolverParameters(max_num_splits=1, max_num_disconnections=0),
)

optimizer_data, _dynamic_stats, _initial_strategy = initialize_optimization(
    params=dc_params,
    optimization_id="example4",
    static_information_files=[grid_file.static_information_file],   # read from processed_gridfile_fs
    processed_gridfile_fs=DirFileSystem(str(OPT_WORK.parent)),
)
print("optimizer reads :", grid_file.static_information_file)
print("workset size    :", optimizer_data.runtime_state.total_workset_size)
print("initial fitness :", round(optimizer_data.initial_fitness, 4))

budget_seconds = 120.0
started = time.monotonic()
epochs, improved = 0, []
while not is_exhausted(optimizer_data) and time.monotonic() - started < budget_seconds:
    optimizer_data = run_epoch(optimizer_data)
    epochs += 1
    improved.extend(optimizer_data.latest_topologies)

print(f"\nepochs run   : {epochs}")
print(f"evaluated    : {get_num_branch_topologies_tried(optimizer_data)}"
      f"/{optimizer_data.runtime_state.total_workset_size}"
      f"  (exhausted={is_exhausted(optimizer_data)})")
print(f"improved     : {len(improved)} topologies in {time.monotonic() - started:.1f}s")

if improved:
    best = max(improved, key=lambda topology: topology.metrics.fitness)
    print(f"\nbest fitness : {best.metrics.fitness:.4f}  (baseline {optimizer_data.initial_fitness:.4f})")
    print(f"actions      : {best.actions}   disconnections: {best.disconnections}")
    print(f"overload_energy_n_1: {best.metrics.extra_scores.get('overload_energy_n_1'):.4f}"
          f"  (baseline {optimizer_data.initial_metrics.get('overload_energy_n_1'):.4f})")

## Stage 4 — AC validation

CA takes the **canonical** definition, not the DC projection: AC can evaluate cases DC cannot.
Before dispatch it derives an in-memory copy — SPPS rules are stripped for imported lists (no engine
executes them yet), and auto-generated switch-only cases are stripped for mask-derived ones. That
copy is never persisted.

In [ ]:
from toop_engine_contingency_analysis.ac_loadflow_service import get_ac_loadflow_results
from toop_engine_grid_helpers.powsybl.loadflow_parameters import CGMES_DISTRIBUTED_SLACK

# What get_ac_loadflow_results derives internally, shown explicitly:
if canonical.source_schema == "complex":
    ac_definition = copy_without_spps_rules(canonical)
else:
    ac_definition = copy_without_switch_only_contingencies(canonical)

print(f"canonical  : {len(canonical.contingencies)} cases, "
      f"spps={[r.scheme_name for r in (canonical.spps_rules or [])]}")
print(f"AC runtime : {len(ac_definition.contingencies)} cases, spps={ac_definition.spps_rules}")

net = pypowsybl.network.load(str(WORK / PREPROCESSING_PATHS["grid_file_path_powsybl"]))
loadflow_results = get_ac_loadflow_results(
    net=net,
    n_minus_1_definition=canonical,   # sanitised inside
    timestep=0,
    lf_params=CGMES_DISTRIBUTED_SLACK,
)
# The polars results come back lazy; collect them to look at the frames.
converged = loadflow_results.converged
branch_results = loadflow_results.branch_results
if hasattr(converged, "collect"):
    converged, branch_results = converged.collect(), branch_results.collect()

print(f"\n{type(loadflow_results).__name__}: "
      f"{converged.height} convergence rows, {branch_results.height} branch rows")
converged.head()

## Recap

- The **importer** owns `nminus1_definition.json`; nothing downstream rewrites it.
- **DC** derives `dc_nminus1_definition.json` — exactly the cases it computes, in solver order — and
  treats it as an output only.
- **CA/AC** derives an in-memory copy and persists nothing.

Each arrow in that chain is a plain `Nminus1Definition`, which is why the hand-overs are inspectable
with nothing more than `load_nminus1_definition`.